# 🌐 **FASE 1 - Algoritmos implementados**

In [1]:
import itertools
import networkx as nx

In [2]:
def ham_path_fuerza_bruta(G):
    """
    Resuelve HAM-PATH por fuerza bruta: enumera TODAS las permutaciones de los
    vértices y comprueba si alguna constituye un camino hamiltoniano.

    Parámetros:
    - G (networkx.Graph): grafo no dirigido de entrada.

    Retorna:
    - (P, R) donde:
        P (list | None): lista [v1, ..., vn] del primer camino hamiltoniano
                         encontrado, o None si no existe.
        R (bool)       : True si G admite un camino hamiltoniano; False si no.
    """
    vertices = list(G.nodes())
    n = len(vertices)

    # Grafo sin vértices: no hay camino que reportar.
    if n == 0:
        return None, False

    # Se examinan las n! ordenaciones posibles de los vértices (de forma perezosa).
    for path in itertools.permutations(vertices):
        # Una permutación es camino hamiltoniano si todo par consecutivo es arista.

        es_camino = True

        for i in range(n-1):
          if not G.has_edge(path[i], path[i+1]):
            es_camino = False
            break

        if es_camino:
            return list(path), True

    # Ninguna permutación resultó ser un camino hamiltoniano.
    return None, False

In [3]:
def ham_path_backtracking(G):
    """
    Resuelve HAM-PATH por backtracking (fuerza bruta con poda).

    Construye el camino incrementalmente y retrocede en cuanto un camino
    parcial no puede prolongarse hacia un vecino no visitado.

    Parámetros:
    - G (networkx.Graph): grafo no dirigido de entrada.

    Retorna:
    - (P, R) donde:
        P (list | None): lista [v1, ..., vn] del camino hamiltoniano hallado,
                         o None si no existe.
        R (bool)       : True si G admite un camino hamiltoniano; False si no.
    """
    n = G.number_of_nodes()

    # Grafo sin vértices: no hay camino que reportar.
    if n == 0:
        return None, False

    vertices = list(G.nodes())

    def backtrack(path, visitados):

        # Caso base: el camino parcial ya cubre los n vértices -> es hamiltoniano.
        if len(path) == n:
            return True

        actual = path[-1]

        # Se intenta extender hacia cada vecino no visitado.
        for v in sorted(G.neighbors(actual)):
            if v not in visitados:
                # Se toma la decisión de avanzar hacia v.
                path.append(v)
                visitados.add(v)

                if backtrack(path, visitados):
                    return True

                # v no condujo a una solución: se deshace la decisión (retroceso/poda).
                path.pop()
                visitados.remove(v)

        # Ningún vecino prolongó el camino: callejón sin salida.
        return False

    # Se prueba cada vértice como posible punto de partida del camino.
    for inicio in vertices:
        path = [inicio]
        visitados = {inicio}
        if backtrack(path, visitados):
            return path, True

    # Ningún inicio produjo un camino hamiltoniano.
    return None, False

In [4]:
def ham_path_rubin(G):
    """
    Busca un camino hamiltoniano en G (networkx.Graph, NO dirigido) mediante el
    procedimiento de Rubin (1974): backtracking con un test de admisibilidad
    basado en reglas de deducción que podan caminos parciales inviables.

    Parámetros:
    - G (networkx.Graph): grafo no dirigido de entrada.

    Retorna:
    - (P, R) donde:
        P (list | None): lista [v1, ..., vn] del camino hamiltoniano hallado,
                         o None si no existe.
        R (bool)       : True si G admite un camino hamiltoniano; False si no.
    """
    n = G.number_of_nodes()
    if n == 0:
        return None, False

    vertices = list(G.nodes())
    sol = [None]   # almacena la solución cuando se encuentra

    def admisible(path, visitados):
        """Test de admisibilidad (paso S2): aplica las reglas de deducción sobre H."""
        last = path[-1]
        pendientes = [u for u in vertices if u not in visitados]
        if not pendientes:
            return True   # camino completo: admisible

        # H = subgrafo inducido por Disp = {last} ∪ pendientes (los que aún
        # pueden formar nuevas aristas del camino).
        disp = set(pendientes)
        disp.add(last)

        # (F1) vértice aislado  y  (F5) a lo sumo un pendiente de grado 1.
        grado_uno = 0
        for u in pendientes:
            grado_u = sum(1 for w in G.neighbors(u) if w in disp and w != u)
            if grado_u == 0:            # (F1) u nunca podrá conectarse
                return False
            if grado_u == 1:            # u solo puede ser un extremo del resto
                grado_uno += 1
                if grado_uno >= 2:      # (F5) tres extremos -> imposible
                    return False

        # (F7) conectividad: todos los pendientes alcanzables desde 'last' en H.
        permitido = set(pendientes)
        permitido.add(last)
        alcanzables = set()
        pila = [last]
        while pila:
            x = pila.pop()
            for w in G.neighbors(x):
                if w in permitido and w not in alcanzables:
                    alcanzables.add(w)
                    pila.append(w)
        return all(u in alcanzables for u in pendientes)

    def buscar(path, visitados):
        # Caso base (S7): el camino cubre los n vértices -> hamiltoniano.
        if len(path) == n:
            sol[0] = list(path)
            return True
        # (S2) Poda por deducción: si el prefijo es inadmisible, se abandona.
        if not admisible(path, visitados):
            return False
        # (S3) Extensión hacia vecinos no visitados (orden fijo -> determinista).
        for v in sorted(G.neighbors(path[-1])):
            if v not in visitados:
                path.append(v); visitados.add(v)
                if buscar(path, visitados):
                    return True
                path.pop(); visitados.remove(v)   # (S4) retroceso
        return False

    # (S1) Se prueba cada vértice como posible inicio del camino.
    for inicio in vertices:
        if buscar([inicio], {inicio}):
            return sol[0], True

    # (S6) Ningún inicio produjo un camino hamiltoniano.
    return None, False

In [5]:
def ham_path_prog_dinamica(G):
    """
    Resuelve HAM-PATH por programación dinámica (algoritmo de Held-Karp) usando
    máscaras de bits para representar los subconjuntos de vértices visitados.

    dp[mask][v] = True si existe un camino simple que visita EXACTAMENTE los
    vértices indicados por 'mask' y TERMINA en el vértice de índice v.

    Parámetros:
    - G (networkx.Graph): grafo no dirigido de entrada.

    Retorna:
    - (P, R) donde:
        P (list | None): lista [v1, ..., vn] del camino hamiltoniano hallado,
                         o None si no existe.
        R (bool)       : True si G admite un camino hamiltoniano; False si no.
    """

    n = G.number_of_nodes()
    if n == 0:
        return None, False

    vertices = list(G.nodes())
    idx = {v: i for i, v in enumerate(vertices)}   # vértice -> índice de bit (0..n-1) (diccionario)
    FULL = (1 << n) - 1                            # máscara del conjunto total V

    # Tablas indexadas por (máscara, índice de vértice final).
    dp     = [[False] * n for _ in range(1 << n)]  # dp[mask][v]  -> ¿estado alcanzable?
    parent = [[-1]    * n for _ in range(1 << n)]  # predecesor para reconstruir el camino

    # --- Caso base: camino de un solo vértice {v_i}, que termina en v_i. ---
    for i in range(n):
        dp[(1 << i)][i] = True

    # --- Llenado de la tabla (recurrencia) ---
    # Se recorren las máscaras en orden creciente: añadir un vértice SIEMPRE
    # aumenta el valor entero de la máscara, así que al procesar 'mask' todos
    # sus subconjuntos con un vértice menos ya fueron resueltos.

    for mask in range(1 << n):
        for v in range(n):
            if not dp[mask][v]:
                continue   # este estado no es alcanzable; nada que propagar

            # Se intenta extender el camino (cubre 'mask', termina en v) hacia
            # un vecino w aún NO visitado.

            for w in G.neighbors(vertices[v]):
                j = idx[w]
                if mask & (1 << j):        # w ya está en el conjunto -> se repetiría
                    continue
                nueva = mask | (1 << j)    # se añade w al conjunto visitado
                if not dp[nueva][j]:
                    dp[nueva][j] = True
                    parent[nueva][j] = v   # se recuerda de dónde vino


    # --- ¿Existe camino hamiltoniano?  dp[FULL][v] para algún v. ---
    for v in range(n):
        if dp[FULL][v]:
            # Reconstrucción del camino siguiendo los punteros hacia atrás.
            path = []
            mask, u = FULL, v

            while u != -1:
                path.append(vertices[u])
                previo = parent[mask][u]
                mask ^= (1 << u)         # se quita 'u' del conjunto
                u = previo

            path.reverse()                 # se reconstruyó al revés
            return path, True

    return None, False

In [6]:
import random

def ham_path_montecarlo(G, num_intentos=1000, seed=None):
    """
    Resuelve HAM-PATH por Monte Carlo: construye caminos al azar con reinicios.
    Error de un solo lado: nunca da un falso positivo (si devuelve un camino,
    es un camino hamiltoniano real); solo puede fallar por falso negativo.

    Parámetros:
    - G (networkx.Graph): grafo no dirigido de entrada.
    - num_intentos (int): número de caminos aleatorios a probar.
    - seed (int | None) : semilla para reproducibilidad del azar.

    Retorna:
    - (P, R) donde:
        P (list | None): camino hamiltoniano hallado, o None si ningún intento tuvo éxito.
        R (bool)       : True si se halló un camino; False en caso contrario.
    """
    n = G.number_of_nodes()

    if n == 0:
        return None, False

    generador = random.Random(seed)          # generador local -> reproducible
    vertices = list(G.nodes())

    for i in range(num_intentos):

        actual = generador.choice(vertices)  # vértice inicial al azar
        path = [actual]
        visitados = {actual}

        # Caminata aleatoria sin repetición.
        while len(path) < n:
            candidatos = [w for w in G.neighbors(actual) if w not in visitados]

            if not candidatos:
                break                  # atasco: se abandona este intento

            actual = generador.choice(candidatos)   # vecino nuevo al azar
            path.append(actual)
            visitados.add(actual)

        if len(path) == n:             # se cubrieron todos los vértices
            return path, True          # respuesta SEGURA (camino real)

    return None, False                 # ningún intento tuvo éxito (posible falso negativo)

# 🧪 **Proyecto Final — Análisis y Diseño de Algoritmos**
## ⚙️ **Fase 2: Experimentación y recolección de datos — El problema del Camino Hamiltoniano**
---

**Autores:** Juan Diego Ramírez Sánchez, Juan Esteban Agudelo Burgos

**Curso:** Análisis y Diseño de Algoritmos (ADA)

<br>

> Este es el **segundo** de los dos notebooks del proyecto. Mientras que el primero
> (`Proyecto_Final_ADA.ipynb`) contiene las implementaciones, la teoría y las pruebas unitarias,
> **este notebook está dedicado a la Fase 2**: construye la infraestructura de medición, ejecuta
> el barrido experimental que compara los cinco enfoques, y realiza el análisis estadístico y la
> generación de figuras. Está pensado para ejecutarse de forma **local** (p. ej. en Visual Studio
> Code) y reproducir por completo los datos del experimento.

## **0. Instrucciones de uso**
---

Este notebook está diseñado para ejecutarse **localmente** de principio a fin (Visual Studio Code
o Jupyter), lo que garantiza tiempos de CPU estables y sin límites de sesión. Para reproducir los
resultados, siga estas indicaciones:

<br>

**▸ Orden de ejecución.** Ejecute las celdas **secuencialmente, de arriba hacia abajo**. Las
secciones posteriores reutilizan objetos y funciones definidos en las anteriores, por lo que
saltarse una celda puede producir errores de nombre no definido.

**▸ Dependencias.** La primera celda de código instala e importa lo necesario
(`networkx`, `pandas`, `matplotlib`, `psutil`). En un entorno local basta con
`pip install networkx pandas matplotlib psutil` (idealmente desde un `requirements.txt`).

**▸ Reproducibilidad.** Todo el azar (la generación de grafos aleatorios y el algoritmo de Monte
Carlo) se deriva de una **semilla global fija** (`SEMILLA = 42`): una nueva ejecución produce
exactamente las mismas instancias y decisiones. Solo los tiempos absolutos variarán según el
hardware, que queda documentado automáticamente en `resultados/entorno.json`.

**▸ Diseño.** La celda del barrido experimental es la única etapa pesada; una vez ejecutada, deja los
datos brutos en `resultados/resultados_crudos.csv`, y el análisis posterior puede repetirse cuantas
veces se quiera sin rehacer el barrido.

> **⚠️ Etapa costosa.** La celda de ejecución del experimento (Sección 5.6) puede tardar varios
> minutos según los topes elegidos. Ajuste los parámetros `repeticiones`, `limite_seg` y `n_max` en la celda de
> configuración según su máquina y el tiempo disponible.

<br>

**▸ Salidas que produce este notebook.**

| Archivo | Contenido |
|---|---|
| `resultados/entorno.json` | Registro del hardware y software de la corrida. |
| `resultados/resultados_crudos.csv` | Datos brutos: una fila por corrida (1155 en total). |
| `resultados/resumen_estadistico.csv` | Resumen agregado por algoritmo, caso y tamaño. |
| `figuras/*.png` | Las 14 figuras del análisis (tiempo, boxplots, memoria, teoría). |

<br>

**▸ Organización del notebook.**

| Sección | Contenido |
|---|---|
| 4 | Configuración del entorno experimental (imports, registro del entorno, directorios). |
| 5 | Infraestructura del experimento (`Timer`, `GrafoContador`, medición y motor del barrido). |
| 6 | Análisis estadístico y visualización (carga de datos, estadística descriptiva, figuras). |

En esta fase medimos empíricamente el rendimiento de los cinco enfoques. Siguiendo las buenas
prácticas de investigación empírica (medición rigurosa, repetición, documentación y
reproducibilidad). Todo el código es portable: corre igual en Colab o en un entorno local; solo cambian *dónde* se
genera la carga pesada y *hasta qué tamaño* $n$ es computacionalmente factible para cada algoritmo.

<br>

## **4. Configuración del entorno experimental**
---

<br>

### **4.1. Instalación e importación de librerías**

Se instalan e importan las librerías necesarias. En Colab casi todas vienen preinstaladas; la
celda de instalación se conserva por reproducibilidad. En un entorno local, basta con
`pip install networkx pandas matplotlib psutil`.

In [7]:
%pip install networkx pandas matplotlib psutil

# --- Librerías estándar ---
import time
import random
import itertools
import json
import platform
import sys
import os
import tracemalloc          # medición de memoria (se usará en la Sección 5)
from pathlib import Path
from datetime import datetime

# --- Librerías de terceros ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

%matplotlib inline

print("Librerías importadas correctamente.")


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Librerías importadas correctamente.


### **4.2. Registro del entorno de ejecución (reproducibilidad)**

Documentamos el hardware (CPU, RAM) y el
software (versión de Python, librerías, sistema operativo) donde se tomaron las mediciones, para
que cualquier investigador pueda replicar el estudio. La función `registrar_entorno` captura esa
información **automáticamente** y la guarda en `resultados/entorno.json`, además de mostrarla como
tabla. Usa `psutil` si está disponible y recurre a un respaldo portable en caso contrario.

<br>

> **Importante.** Como los datos se generarán en un entorno local, este registro debe ejecutarse
> **en esa misma máquina**, de modo que refleje el hardware donde realmente se midió.

In [8]:
def registrar_entorno(guardar_en=None):
    """
    Captura el hardware y el software del entorno de ejecución para documentar
    la reproducibilidad del experimento.

    Parámetros:
    - guardar_en (str | Path | None): ruta de un JSON donde volcar el registro.

    Retorna:
    - dict con los parámetros del entorno.
    """

    # RAM y CPU: se usa psutil si está disponible; si no, un respaldo portable.
    try:
        import psutil
        ram_gb = round(psutil.virtual_memory().total / (1024**3), 2)
        nucleos_fisicos = psutil.cpu_count(logical=False)
        nucleos_logicos = psutil.cpu_count(logical=True)
    except Exception:
        ram_gb = "no disponible (instale psutil)"
        nucleos_fisicos = "?"
        nucleos_logicos = os.cpu_count()

    entorno = {
        "fecha_registro":    datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "sistema_operativo": f"{platform.system()} {platform.release()}",
        "arquitectura":      platform.machine(),
        "procesador":        platform.processor() or "no reportado",
        "nucleos_fisicos":   nucleos_fisicos,
        "nucleos_logicos":   nucleos_logicos,
        "ram_total_gb":      ram_gb,
        "python":            platform.python_version(),
        "numpy":             np.__version__,
        "pandas":            pd.__version__,
        "matplotlib":        matplotlib.__version__ if (matplotlib := __import__("matplotlib")) else "?",
        "networkx":          nx.__version__,
    }

    if guardar_en is not None:
        Path(guardar_en).write_text(json.dumps(entorno, indent=2, ensure_ascii=False))

    return entorno

### **4.3. Directorios de salida y configuración global**

Se crean los directorios donde se almacenarán los resultados brutos (`resultados/`) y las figuras
(`figuras/`), y se fija una **semilla global** (`SEMILLA`) para que la generación de instancias
aleatorias y el algoritmo de Monte Carlo sean **reproducibles**. Con el entorno ya definido, se
registra y se muestra en pantalla.

In [9]:
# --- Directorios de salida (rutas relativas: portables entre Colab y local) ---
RESULTADOS_DIR = Path("resultados")
FIGURAS_DIR    = Path("figuras")
RESULTADOS_DIR.mkdir(exist_ok=True)
FIGURAS_DIR.mkdir(exist_ok=True)

# --- Semilla global para reproducibilidad ---
SEMILLA = 42

# --- Registro del entorno de ejecución ---
entorno = registrar_entorno(guardar_en=RESULTADOS_DIR / "entorno.json")

print(f"Directorios listos: '{RESULTADOS_DIR}/' y '{FIGURAS_DIR}/'")
print(f"Semilla global: {SEMILLA}\n")
print("Entorno de ejecución registrado en 'resultados/entorno.json':")
pd.DataFrame(list(entorno.items()), columns=["Parámetro", "Valor"])

Directorios listos: 'resultados/' y 'figuras/'
Semilla global: 42

Entorno de ejecución registrado en 'resultados/entorno.json':


,Parámetro,Valor
0,fecha_registro,2026-07-14 03:36:20
1,sistema_operativo,Windows 11
2,arquitectura,AMD64
3,procesador,"AMD64 Family 23 Model 104 Stepping 1, Authenti..."
4,nucleos_fisicos,4
5,nucleos_logicos,8
6,ram_total_gb,7.33
7,python,3.13.3
8,numpy,2.3.1
9,pandas,3.0.3


## **5. Infraestructura del experimento**
---

Esta sección construye las herramientas de medición y el motor que ejecuta el experimento con
rigor estadístico. Las **variables independientes** que controlamos son el *algoritmo*, el
*tamaño de entrada* $n$ y el *tipo de caso* (mejor/peor/promedio); las **variables dependientes**
que medimos son el *tiempo de ejecución*, el *número de operaciones básicas* y el *uso de memoria*.

<br>

### **5.1. Medición de tiempo: la clase `Timer` como context manager**

Para medir el tiempo real usamos `time.perf_counter()`, el reloj de mayor
resolución de Python y el recomendado para *benchmarking*. Siguiendo la guía de Real Python
(https://realpython.com/python-timer/), encapsulamos la medición en una clase `Timer` y la
**extendemos con el protocolo de *context manager*** (`__enter__`/`__exit__`), de modo que medir
un bloque sea tan simple como `with Timer() as t: ...`. La clase también valida usos indebidos
mediante una excepción propia, `TimerError`.

In [10]:
class TimerError(Exception):
    """Excepción para reportar usos indebidos de la clase Timer."""


class Timer:
    """
    Cronómetro basado en time.perf_counter(), utilizable como context manager.
    Inspirado en la guía de Real Python (https://realpython.com/python-timer/).

    Uso:
        with Timer() as t:
            ... código a medir ...
        print(t.ultimo)   # tiempo transcurrido en segundos
    """
    def __init__(self):
        self._inicio = None
        self.ultimo = None          # último tiempo medido (segundos)

    def start(self):
        """Inicia el cronómetro."""
        if self._inicio is not None:
            raise TimerError("El cronómetro ya está en marcha. Use .stop() primero.")
        self._inicio = time.perf_counter()

    def stop(self):
        """Detiene el cronómetro y devuelve el tiempo transcurrido."""
        if self._inicio is None:
            raise TimerError("El cronómetro no está en marcha. Use .start() primero.")
        self.ultimo = time.perf_counter() - self._inicio
        self._inicio = None
        return self.ultimo

    # --- Protocolo de context manager (extensión de la clase base) ---
    def __enter__(self):
        self.start()
        return self

    def __exit__(self, *exc_info):
        self.stop()

### **5.2. Generación de instancias por tipo de caso**

Para cada tamaño $n$ generamos instancias de tres tipos, que corresponden al análisis de casos:

- **Mejor caso** → grafo **completo** $K_n$: siempre tiene camino y se halla de inmediato.
- **Peor caso** → grafo **bipartito completo desbalanceado** $K_{a,b}$ con $\lvert a-b\rvert \geq 2$:
  provablemente **sin** camino hamiltoniano (en un grafo bipartito el camino debe alternar lados,
  así que las partes deben diferir en a lo sumo $1$), lo que obliga a los algoritmos exactos a
  **agotar la búsqueda** antes de concluir "no".
- **Caso promedio** → grafo **aleatorio** $G(n, 0.5)$: comportamiento típico, con respuesta
  variable. La semilla garantiza reproducibilidad.

In [11]:
import math

def generar_instancia(n, tipo, semilla=None):
    """
    Genera una instancia (networkx.Graph) de tamaño n según el tipo de caso.

    - "mejor"   : grafo completo K_n (con camino, hallado de inmediato).
    - "peor"    : bipartito completo desbalanceado (sin camino hamiltoniano).
    - "promedio": grafo aleatorio G(n, 0.5).
    """
    
    if tipo == "mejor":
        return nx.complete_graph(n)
    if tipo == "peor":
        a = min(math.ceil(n / 2) + 1, n)   # parte mayor
        b = n - a                          # parte menor  (|a - b| >= 2)
        return nx.complete_bipartite_graph(a, b)
    if tipo == "promedio":
        return nx.gnp_random_graph(n, 0.5, seed=semilla)
    raise ValueError(f"Tipo de caso desconocido: {tipo}")

### **5.3. Conteo de operaciones básicas**

Como **operación básica** común a los cinco algoritmos adoptamos la **consulta de adyacencia**
(inspeccionar si dos vértices son vecinos, o recorrer la vecindad de un vértice): es el primitivo
que todos comparten y el que domina su costo. Para contarlo **sin modificar** los algoritmos ya
validados, se define `GrafoContador`, un envoltorio que delega en el grafo real pero **incrementa
un contador** en cada `has_edge` y en cada vecino recorrido con `neighbors`. Basta con ejecutar el
algoritmo sobre el grafo envuelto y leer su atributo `operaciones`.

<br>

> Nótese que esta métrica mide el **trabajo primitivo total**, incluyendo la sobrecarga de las
> podas: por eso un algoritmo que explora *menos nodos* (como Rubin) puede, aun así, realizar
> *más* consultas de adyacencia por nodo.

In [12]:
class GrafoContador:
    """
    Envuelve un grafo de networkx y cuenta las consultas de adyacencia
    (has_edge y cada vecino recorrido con neighbors) como 'operaciones básicas'.
    Delega el resto de atributos en el grafo original, de modo que los algoritmos
    se ejecutan SIN modificación.
    """
    
    def __init__(self, G):
        self._G = G
        self.operaciones = 0

    def has_edge(self, u, v):
        self.operaciones += 1
        return self._G.has_edge(u, v)

    def neighbors(self, v):
        for w in self._G.neighbors(v):
            self.operaciones += 1     # cada vecino recorrido cuenta como una operación
            yield w

    def number_of_nodes(self):
        return self._G.number_of_nodes()

    def nodes(self):
        return self._G.nodes()

    def __getattr__(self, nombre):
        return getattr(self._G, nombre)   # delega todo lo demás

### **5.4. Medición de una corrida**

La función `medir_una_corrida` mide las tres variables dependientes para una ejecución. Es
importante **separar** las mediciones: el tiempo se toma en una corrida **limpia** (sin
instrumentación), porque tanto el contador de operaciones como `tracemalloc` introducen
sobrecarga que **inflaría** el tiempo. Por eso el algoritmo se ejecuta tres veces:

1. una corrida cronometrada con `Timer` (tiempo puro);
2. una corrida sobre `GrafoContador` (operaciones básicas);
3. una corrida bajo `tracemalloc` para el **pico** de memoria.

In [13]:
def medir_una_corrida(alg, G):
    """
    Mide tiempo, operaciones básicas y pico de memoria de una ejecución de 'alg' sobre 'G'.
    Cada métrica se toma en una corrida independiente para no contaminar las demás.

    Retorna un dict con: tiempo (s), operaciones, memoria_kb, encontrado (bool), long_camino.
    """
    
    # (1) Tiempo: corrida limpia, sin instrumentación.
    with Timer() as cronometro:
        P, R = alg(G)
    tiempo = cronometro.ultimo

    # (2) Operaciones básicas: corrida sobre el grafo envuelto.
    G_contador = GrafoContador(G)
    alg(G_contador)
    operaciones = G_contador.operaciones

    # (3) Memoria: corrida bajo tracemalloc (pico de asignación).
    tracemalloc.start()
    alg(G)
    _, pico = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    return {
        "tiempo": tiempo,
        "operaciones": operaciones,
        "memoria_kb": pico / 1024,
        "encontrado": R,
        "long_camino": len(P) if P else 0,
    }

### **5.5. Motor del experimento y volcado a CSV**

`ejecutar_experimento` recorre el producto **algoritmo × tipo de caso × tamaño**, repitiendo cada
configuración `repeticiones` veces para obtener consistencia estadística, y **almacena una fila
por repetición** (datos *brutos*, que la Sección 6 agregará). Incorpora dos mecanismos para escalar hasta donde sea computacionalmente factible:

- un **tope de tamaño por algoritmo** (`n_max`), como salvaguarda frente al agotamiento de memoria
  (sobre todo en la programación dinámica);
- un **corte por presupuesto de tiempo** (`limite_seg`): si el tiempo promedio de una configuración
  lo supera, se deja de aumentar $n$ para ese algoritmo/tipo.

Al terminar, vuelca todo a `resultados/resultados_crudos.csv`.

In [14]:
def ejecutar_experimento(algoritmos, tamanos, tipos, repeticiones=5,
                         limite_seg=5, semilla=SEMILLA, n_max=None,
                         archivo_csv=RESULTADOS_DIR / "resultados_crudos.csv"):
    """
    Ejecuta el barrido experimental y vuelca los resultados brutos a un CSV.

    SONDEO por tamaño: se hace una corrida de prueba
    antes de comprometer las réplicas; si ya supera 'limite_seg', se corta el
    escalado de inmediato (evita ejecutar decenas de corridas lentísimas).

    Parámetros:
    - algoritmos (dict): {nombre: función(G) -> (P, R)}.
    - tamanos (iterable): tamaños de entrada n a barrer.
    - tipos (list): subconjunto de {"mejor", "peor", "promedio"}.
    - repeticiones (int): réplicas por configuración.
    - limite_seg (float): corte por presupuesto de tiempo promedio.
    - semilla (int): semilla base para reproducibilidad.
    - n_max (dict | None): tope de n por algoritmo (salvaguarda de memoria).
    - archivo_csv (Path): ruta de salida.

    Retorna: DataFrame con los datos brutos (una fila por repetición).    
    """
    filas = []
    n_max = n_max or {}

    for nombre, alg in algoritmos.items():
        for tipo in tipos:
            for n in tamanos:
                if n > n_max.get(nombre, 10**9):
                    break
                if tipo == "peor" and n < 4:
                    continue

                # --- SONDEO: una sola corrida antes de comprometer las réplicas ---
                G_sonda = generar_instancia(n, tipo, semilla=semilla + n)
                with Timer() as t_sonda:
                    alg(G_sonda)
                if t_sonda.ultimo > limite_seg:
                    print(f"  [corte] {nombre}/{tipo}: n={n} tardó {t_sonda.ultimo:.1f}s "
                          f"(> {limite_seg}s). Se detiene el escalado.")
                    break

                # --- Medición completa de las réplicas ---
                for rep in range(repeticiones):
                    G = generar_instancia(n, tipo, semilla=semilla + rep * 997 + n)
                    met = medir_una_corrida(alg, G)
                    filas.append({
                        "algoritmo":   nombre,
                        "tipo_caso":   tipo,
                        "n":           n,
                        "m":           G.number_of_edges(),
                        "repeticion":  rep,
                        "tiempo_seg":  met["tiempo"],
                        "operaciones": met["operaciones"],
                        "memoria_kb":  met["memoria_kb"],
                        "encontrado":  met["encontrado"],
                        "long_camino": met["long_camino"],
                    })

    df = pd.DataFrame(filas)
    df.to_csv(archivo_csv, index=False)
    print(f"\nExperimento terminado: {len(df)} filas guardadas en '{archivo_csv}'.")
    return df

### **5.6. Ejecución del experimento**

Se define la configuración y se lanza el barrido. Los algoritmos se registran con una **interfaz
uniforme** `función(G) -> (P, R)`; el de Monte Carlo se adapta fijando sus parámetros con una
`lambda`. Los topes `n_max` reflejan hasta dónde es razonable llevar cada enfoque (la fuerza bruta
muere pronto por tiempo; la programación dinámica, por memoria; Monte Carlo escala mucho más).

<br>

> **⚠️ Etapa costosa.** Esta celda es la que conviene ejecutar en el **entorno local** (VS Code).
> Puede tardar según los topes elegidos. Al finalizar, deja `resultados/resultados_crudos.csv`
> listo para el análisis de la Sección 6. Ajuste `repeticiones`, `limite_seg` y `n_max` según su
> máquina y el tiempo disponible.

In [ ]:
algoritmos = {
    "fuerza_bruta":  ham_path_fuerza_bruta,
    "backtracking":  ham_path_backtracking,
    "rubin":         ham_path_rubin,
    "prog_dinamica": ham_path_prog_dinamica,
    "monte_carlo":   lambda G: ham_path_montecarlo(G, num_intentos=2000, seed=SEMILLA),
}

# Topes realistas: el sondeo corta antes en el peor caso, pero acotamos por si acaso.
n_max_por_algoritmo = {
    "fuerza_bruta":  11,    # ~12! ya es prohibitivo
    "backtracking":  22,    # el sondeo lo detendrá en el peor caso ~n=13
    "rubin":         22,
    "prog_dinamica": 20,    # n=21 ≈ 40 s; límite de memoria (O(2^n · n))
    "monte_carlo":   26,    # escala de forma polinómica con num_intentos fijo
}

TAMANOS      = list(range(4, 27))
TIPOS        = ["mejor", "peor", "promedio"]
REPETICIONES = 5       # réplicas por configuración
LIMITE_SEG   = 5       # corte por presupuesto de tiempo

df_crudo = ejecutar_experimento(
    algoritmos, TAMANOS, TIPOS,
    repeticiones=REPETICIONES,
    limite_seg=LIMITE_SEG,
    semilla=SEMILLA,
    n_max=n_max_por_algoritmo,
)
df_crudo.head()

  [corte] fuerza_bruta/peor: n=11 tardó 18.8s (> 5s). Se detiene el escalado.
  [corte] backtracking/peor: n=13 tardó 5.4s (> 5s). Se detiene el escalado.
  [corte] rubin/peor: n=13 tardó 7.7s (> 5s). Se detiene el escalado.
  [corte] prog_dinamica/mejor: n=18 tardó 6.9s (> 5s). Se detiene el escalado.
  [corte] prog_dinamica/peor: n=20 tardó 9.4s (> 5s). Se detiene el escalado.
  [corte] prog_dinamica/promedio: n=19 tardó 9.5s (> 5s). Se detiene el escalado.

Experimento terminado: 1155 filas guardadas en 'resultados\resultados_crudos.csv'.


,algoritmo,tipo_caso,n,m,repeticion,tiempo_seg,operaciones,memoria_kb,encontrado,long_camino
0,fuerza_bruta,mejor,4,6,0,0.000020,3,0.3125,True,4
1,fuerza_bruta,mejor,4,6,1,0.000019,3,0.3125,True,4
2,fuerza_bruta,mejor,4,6,2,0.000008,3,0.3125,True,4
3,fuerza_bruta,mejor,4,6,3,0.000006,3,0.3125,True,4
4,fuerza_bruta,mejor,4,6,4,0.000007,3,0.3125,True,4


## **6. Análisis estadístico y visualización**
---

Esta sección carga los datos brutos generados en la Sección 5, calcula resúmenes estadísticos con
Pandas y produce las gráficas que sustentan el experimento.

<br>

### **6.1. Métodos de recolección utilizados**

Los datos provienen de la infraestructura de la Sección 5, que registró tres variables
dependientes con módulos distintos: el **tiempo** con la clase `Timer` (basada en
`time.perf_counter`), el **número de operaciones básicas** con el envoltorio `GrafoContador`
(que cuenta consultas de adyacencia), y el **pico de memoria** con el módulo `tracemalloc` de la
biblioteca estándar. Cada configuración *(algoritmo, tipo de caso, tamaño)* se repitió varias
veces, y cada réplica quedó como una fila en `resultados/resultados_crudos.csv`.

### **6.2. Estadística descriptiva con Pandas**

Se agrupan los datos por *(algoritmo, tipo de caso, tamaño)* y se calculan **medidas de tendencia
central** (media, mediana y moda) y **de dispersión** (desviación estándar, mínimo y máximo) del
tiempo, junto con la memoria media y la moda del número de operaciones. La moda es más informativa
sobre variables discretas (como las operaciones, deterministas en los algoritmos exactos) que
sobre el tiempo, que es continuo. El resumen se guarda en `resultados/resumen_estadistico.csv`.

In [17]:
# --- Carga de los datos brutos ---
df = pd.read_csv(RESULTADOS_DIR / "resultados_crudos.csv")
print(f"Datos cargados: {len(df)} filas.")

# --- Moda robusta (Series.mode puede devolver varios valores) ---
def moda_segura(serie):
    m = serie.mode()
    return m.iloc[0] if not m.empty else np.nan

# --- Resumen estadístico agrupado ---
resumen = (
    df.groupby(["algoritmo", "tipo_caso", "n"])
      .agg(tiempo_medio    =("tiempo_seg", "mean"),
           tiempo_mediana  =("tiempo_seg", "median"),
           tiempo_std      =("tiempo_seg", "std"),
           tiempo_min      =("tiempo_seg", "min"),
           tiempo_max      =("tiempo_seg", "max"),
           operaciones_moda=("operaciones", moda_segura),
           memoria_media   =("memoria_kb", "mean"))
      .reset_index()
)

resumen.to_csv(RESULTADOS_DIR / "resumen_estadistico.csv", index=False)
print(f"Resumen guardado en 'resultados/resumen_estadistico.csv' ({len(resumen)} filas).")
resumen.head(10)

Datos cargados: 1155 filas.
Resumen guardado en 'resultados/resumen_estadistico.csv' (231 filas).


,algoritmo,tipo_caso,n,tiempo_medio,tiempo_mediana,tiempo_std,tiempo_min,tiempo_max,operaciones_moda,memoria_media
0,backtracking,mejor,4,0.000008,0.000007,3.010483e-06,0.000005,0.000013,9,1.057813
1,backtracking,mejor,5,0.000006,0.000006,1.788868e-07,0.000005,0.000006,16,1.570312
2,backtracking,mejor,6,0.000007,0.000007,2.701844e-07,0.000006,0.000007,25,1.820312
3,backtracking,mejor,7,0.000007,0.000007,1.923553e-07,0.000007,0.000008,36,1.929688
4,backtracking,mejor,8,0.000008,0.000008,1.516587e-07,0.000008,0.000008,49,2.132812
5,backtracking,mejor,9,0.000035,0.000009,5.782537e-05,0.000009,0.000138,64,2.220312
6,backtracking,mejor,10,0.000010,0.000010,2.701841e-07,0.000009,0.000010,81,2.507812
7,backtracking,mejor,11,0.000011,0.000011,2.863559e-07,0.000010,0.000011,100,2.648438
8,backtracking,mejor,12,0.000017,0.000020,5.251192e-06,0.000011,0.000022,121,2.945312
9,backtracking,mejor,13,0.000017,0.000018,3.737380e-06,0.000013,0.000022,144,3.101562


### **6.3. Estilo común de las gráficas**

Se fija un **color y un marcador por algoritmo** (consistentes en todas las figuras) y una función
auxiliar `guardar_figura` que exporta cada gráfico a la carpeta `figuras/`.

In [19]:
ALGORITMOS_ORDEN = ["fuerza_bruta", "backtracking", "rubin", "prog_dinamica", "monte_carlo"]

COLOR = {"fuerza_bruta": "#E63946", "backtracking": "#457B9D", "rubin": "#2A9D8F",
         "prog_dinamica": "#E9C46A", "monte_carlo": "#9B5DE5"}
MARCADOR = {"fuerza_bruta": "o", "backtracking": "s", "rubin": "^",
            "prog_dinamica": "D", "monte_carlo": "v"}

def guardar_figura(fig, nombre):
    """Guarda una figura en la carpeta figuras/ y cierra el objeto."""
    ruta = FIGURAS_DIR / nombre
    fig.savefig(ruta, dpi=120, bbox_inches="tight")
    plt.close(fig)
    print(f"  figura guardada: {ruta}")

### **6.4. Tiempo de ejecución vs. tamaño de entrada**

Gráfica central del experimento: el tiempo medio frente a $n$, con **una figura por tipo de caso**
y **escala logarítmica** en el eje del tiempo (los tiempos abarcan varios órdenes de magnitud).
Aquí se aprecia el muro exponencial: la fuerza bruta se dispara primero, la programación dinámica
resiste hasta tamaños mayores, y Monte Carlo se mantiene plano.

In [20]:
for tipo in ["mejor", "peor", "promedio"]:
    fig, ax = plt.subplots(figsize=(7, 4.5))
    sub = resumen[resumen.tipo_caso == tipo]
    for alg in ALGORITMOS_ORDEN:
        d = sub[sub.algoritmo == alg].sort_values("n")
        if d.empty:
            continue
        ax.plot(d.n, d.tiempo_medio, marker=MARCADOR[alg], color=COLOR[alg],
                label=alg, markersize=5, linewidth=1.4)
    ax.set_yscale("log")
    ax.set_xlabel("Tamaño de entrada n")
    ax.set_ylabel("Tiempo medio de ejecución (s, escala log)")
    ax.set_title(f"Tiempo vs. tamaño de entrada — caso {tipo}")
    ax.grid(True, alpha=0.3, which="both")
    ax.legend(fontsize=8)
    guardar_figura(fig, f"tiempo_vs_n_{tipo}.png")

  figura guardada: figuras\tiempo_vs_n_mejor.png
  figura guardada: figuras\tiempo_vs_n_peor.png
  figura guardada: figuras\tiempo_vs_n_promedio.png


### **6.5. Variación entre réplicas (boxplots)**

Para evaluar la **estabilidad de las mediciones**, se muestran boxplots del tiempo entre réplicas
para cada tamaño, con una figura por algoritmo (caso promedio). Cajas estrechas indican mediciones
consistentes; cajas anchas o muchos valores atípicos señalan ruido en la medición.

In [21]:
for alg in ALGORITMOS_ORDEN:
    d = df[(df.algoritmo == alg) & (df.tipo_caso == "promedio")]
    if d.empty:
        continue
    tamanos = sorted(d.n.unique())
    datos = [d[d.n == n].tiempo_seg.values for n in tamanos]

    fig, ax = plt.subplots(figsize=(7.5, 4))
    ax.boxplot(datos)                              # sin 'labels' (portable entre versiones)
    ax.set_xticks(range(1, len(tamanos) + 1))
    ax.set_xticklabels(tamanos)
    ax.set_yscale("log")
    ax.set_xlabel("Tamaño de entrada n")
    ax.set_ylabel("Tiempo (s, escala log)")
    ax.set_title(f"Variación entre réplicas — {alg} (caso promedio)")
    ax.grid(True, alpha=0.3, axis="y")
    guardar_figura(fig, f"boxplot_tiempo_{alg}.png")

  figura guardada: figuras\boxplot_tiempo_fuerza_bruta.png
  figura guardada: figuras\boxplot_tiempo_backtracking.png
  figura guardada: figuras\boxplot_tiempo_rubin.png
  figura guardada: figuras\boxplot_tiempo_prog_dinamica.png
  figura guardada: figuras\boxplot_tiempo_monte_carlo.png


### **6.6. Uso de memoria vs. tamaño de entrada**

Gráfica de barras/bigotes del pico de memoria frente a $n$ (escala logarítmica), con los bigotes
representando la desviación estándar. Aquí debe hacerse evidente el **intercambio espacio–tiempo**:
la programación dinámica crece de forma exponencial en memoria, mientras que los demás enfoques se
mantienen en consumo polinómico.

In [22]:
mem = (df.groupby(["algoritmo", "n"])
         .agg(mem_media=("memoria_kb", "mean"), mem_std=("memoria_kb", "std"))
         .reset_index())

fig, ax = plt.subplots(figsize=(7.5, 4.5))
for alg in ALGORITMOS_ORDEN:
    d = mem[mem.algoritmo == alg].sort_values("n")
    if d.empty:
        continue
    ax.errorbar(d.n, d.mem_media, yerr=d.mem_std.fillna(0), marker=MARCADOR[alg],
                color=COLOR[alg], label=alg, capsize=3, markersize=4, linewidth=1.3)
ax.set_yscale("log")
ax.set_xlabel("Tamaño de entrada n")
ax.set_ylabel("Pico de memoria (kB, escala log)")
ax.set_title("Uso de memoria vs. tamaño de entrada")
ax.grid(True, alpha=0.3, which="both")
ax.legend(fontsize=8)
guardar_figura(fig, "memoria_vs_n.png")

  figura guardada: figuras\memoria_vs_n.png


##### **6.7. Contraste con la complejidad teórica**

Como cierre, se contrasta el tiempo **medido** de **cada algoritmo** con su cota de complejidad
**teórica** del peor caso. Dado que la complejidad solo describe la *forma* del crecimiento (no
segundos absolutos), se ajusta un único factor de escala $c$ —la media geométrica de la razón
$\text{tiempo}/f(n)$ sobre los puntos medidos— de modo que la curva teórica $c\cdot f(n)$ quede
alineada con los datos. Esa misma curva se **prolonga más allá del rango medido** (hasta
$n=30$; la zona extrapolada aparece sombreada) para que su **comportamiento asintótico** se vea
con claridad, aunque la máquina no haya alcanzado a medir esos tamaños.

Se usa el **caso peor** porque es donde se manifiesta la complejidad del peor caso: en los
algoritmos de búsqueda el factor $n!$ solo aflora sin solución que hallar; la programación
dinámica es insensible a la instancia; y Monte Carlo, al no encontrar camino, agota todos sus
intentos. Las complejidades usadas son: fuerza bruta y backtracking, $O(n\cdot n!)$; Rubin,
$O((n+m)\cdot n!)$; programación dinámica, $O(2^{n} n^{2})$; y Monte Carlo, $O(n+m)$ con número de
intentos fijo. Una buena superposición confirma la teoría; una **divergencia** (p. ej. si el
backtracking o Rubin crecen por debajo de $n!$ gracias a la poda) es en sí misma un hallazgo a
discutir.

In [24]:
# --- Complejidad teórica del peor caso, como función de n, por algoritmo ---
def _m_peor(n):
    """Número de aristas de la instancia del peor caso (bipartito) de tamaño n."""
    a = min(math.ceil(n / 2) + 1, n)
    b = n - a
    return a * b

def complejidad_teorica(alg, n):
    if alg in ("fuerza_bruta", "backtracking"):
        return n * math.factorial(n)                    # O(n · n!)
    if alg == "rubin":
        return (n + _m_peor(n)) * math.factorial(n)     # O((n + m) · n!)
    if alg == "prog_dinamica":
        return (2.0 ** n) * (n ** 2)                    # O(2^n · n²)
    if alg == "monte_carlo":
        return float(n + _m_peor(n))                    # O(n + m), con num_intentos fijo
    return float(n)

ETIQUETA_TEO = {
    "fuerza_bruta":  r"$n\cdot n!$",
    "backtracking":  r"$n\cdot n!$",
    "rubin":         r"$(n+m)\cdot n!$",
    "prog_dinamica": r"$2^{n}\,n^{2}$",
    "monte_carlo":   r"$n+m$",
}

N_TEO_MAX = 30          # la curva teórica se prolonga hasta aquí (más allá de lo medido)
CASO_CONTRASTE = "peor"

for alg in ALGORITMOS_ORDEN:
    d = resumen[(resumen.algoritmo == alg) &
                (resumen.tipo_caso == CASO_CONTRASTE)].sort_values("n")
    if d.empty:
        continue

    # Factor de escala c: media geométrica de la razón tiempo_medido / f(n).
    razones = [math.log(t) - math.log(complejidad_teorica(alg, int(nn)))
               for nn, t in zip(d.n, d.tiempo_medio) if t > 0]
    c = math.exp(sum(razones) / len(razones))

    # Curva teórica EXTENDIDA: desde el n mínimo medido hasta N_TEO_MAX.
    n_ini = int(d.n.min())
    ns_teo = list(range(n_ini, N_TEO_MAX + 1))
    y_teo = [c * complejidad_teorica(alg, n) for n in ns_teo]

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(d.n, d.tiempo_medio, "o", color=COLOR[alg],
            label=f"{alg} (medido)", markersize=6)
    ax.plot(ns_teo, y_teo, "--k",
            label=f"{ETIQUETA_TEO[alg]} (teórico, escalado)")
    # Sombrea la zona extrapolada (no medida).
    ax.axvspan(d.n.max(), N_TEO_MAX, color="gray", alpha=0.08,
               label="zona no medida")

    ax.set_yscale("log")
    ax.set_xlabel("Tamaño de entrada n")
    ax.set_ylabel("Tiempo (s, escala log)")
    ax.set_title(f"Real vs. teórico — {alg} (caso {CASO_CONTRASTE})")
    ax.grid(True, alpha=0.3, which="both")
    ax.legend(fontsize=8)
    guardar_figura(fig, f"real_vs_teorico_{alg}.png")

  figura guardada: figuras\real_vs_teorico_fuerza_bruta.png
  figura guardada: figuras\real_vs_teorico_backtracking.png
  figura guardada: figuras\real_vs_teorico_rubin.png
  figura guardada: figuras\real_vs_teorico_prog_dinamica.png
  figura guardada: figuras\real_vs_teorico_monte_carlo.png
